## <h3 align="center"> __Johns Hopkins University__</h3>
## <h3 align="center">__Whiting School of Engineering__</h3>
## <h3 align="center">__Engineering for Professionals__</h3>
## <h3 align="center">__685.801 Data Science: Independent Study__</h3>
## <h3 align="center">__Test Set Model Validation__</h3>

---

- **Purpose:** Load trained checkpoints from `./models`, evaluate every model on the held-out test set, and save each evaluation to `test_{model_name}.json`.
- **Primary test metrics:** `Test Micro F1`, `Test Macro F1`, and `Test Weighted F1`.
- **Additional analysis:** backbone-level summaries, paired backbone deltas, bootstrap pairwise model comparisons, and separate figures.
- **How to use:**
  1. If running in Colab, set `os.environ['IS_COLAB'] = '1'` before the setup cell, or edit the setup cell directly.
  2. Run the notebook top to bottom.
  3. In Colab, the notebook will mount Google Drive for results output and clone the GitHub repo to access `./models` automatically.
  4. Review the saved JSON outputs, summary tables, bootstrap comparisons, and figures.

---

In [ ]:
%pip install numpy pandas scipy seaborn matplotlib scikit-learn statsmodels kagglehub torch torchvision torchao torchmetrics

In [ ]:
# Environment check
import sys, platform
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
try:
    import numpy as np, pandas as pd, scipy, seaborn, matplotlib, sklearn, statsmodels, torch, torchvision, kagglehub
    print(f'numpy:      {np.__version__}')
    print(f'pandas:     {pd.__version__}')
    print(f'scipy:      {scipy.__version__}')
    print(f'seaborn:    {seaborn.__version__}')
    print(f'matplotlib: {matplotlib.__version__}')
    print(f'sklearn:    {sklearn.__version__}')
    print(f'statsmodels:{statsmodels.__version__}')
    print(f'torch:      {torch.__version__}')
    print(f'torchvision:{torchvision.__version__}')
    print(f'kagglehub:  {kagglehub.__version__}')
except Exception as exc:
    print('Optional packages missing or version check failed:', exc)
    print('Use the command in the cell above to install necessary packages.')

In [ ]:
import os
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

REPO_URL = "https://github.com/ddimpfel/JHU_IS_26.git"
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

if IN_COLAB:
    drive.mount('/content/drive')
    repo_root = Path('/content') / REPO_NAME
    if not repo_root.exists():
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    CURRENT_WORK_DIR = repo_root
    os.chdir(CURRENT_WORK_DIR)
    RESULTS_DIR = Path('/content/drive/MyDrive/is26_models/joint_embedding_results')
else:
    CURRENT_WORK_DIR = Path.cwd()

In [ ]:
import json
import os
import re
import random
import subprocess
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from IPython.display import display
from PIL import Image
from scipy.stats import wilcoxon
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from statsmodels.stats.multitest import multipletests
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset

from continual_learning import CILComputerVisionModel, bootstrap_performance_diff
from gre_model_base import GeneralistRouterExperts, cost_proxy, expected_calibration_error, expert_metrics, router_entropy
from joint_embedding import JointEmbeddingModule

## Config

In [ ]:
SEED = 7
NUM_WORKERS = 0
PIN_MEMORY = bool(IN_COLAB)
PERSIST_WORKERS = False
TOTAL_CLASSES = 9
IMG_SIZE = 224
BATCH_SIZE = 64 if IN_COLAB else 8
NUM_TASKS = 3
BS_RESAMPLES = 1000
USE_CLASS_MASKING = True

NUM_EXPERTS = 6
HIDDEN_EXPERT_SIZE = 128
DROPOUT = 0.1
TOP_K = 2
LAMBDA_AUX = 0.05
TFE_D_MODEL = 32
TFE_NHEAD = 4

JE_EMBEDDING_DIM = 256
JE_PROJECTION_DIM = 256
JE_FEATURE_KEY = 'projections'
JE_TEMPERATURE = 0.07

MODELS_DIR = CURRENT_WORK_DIR / 'models'
TEST_RESULTS_DIR = (
    Path('/content/drive/MyDrive/is26_models/test_results')
    if IN_COLAB else CURRENT_WORK_DIR / 'results' / 'test'
)
BACKBONE_ORDER = ['MobileNet', 'ConvNeXt', 'JE MobileNet', 'JE ConvNeXt']
BASELINE_BACKBONE = 'MobileNet'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
np_rng = np.random.default_rng(seed=SEED)
tor_gen = torch.Generator(device='cpu').manual_seed(SEED)

if str(CURRENT_WORK_DIR) not in sys.path:
    sys.path.insert(0, str(CURRENT_WORK_DIR))

TEST_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Using device: {device}')
print(f'IN_COLAB={IN_COLAB}')
print(f'CURRENT_WORK_DIR={CURRENT_WORK_DIR}')
print(f'MODELS_DIR={MODELS_DIR.resolve()}')
print(f'TEST_RESULTS_DIR={TEST_RESULTS_DIR.resolve()}')

## Test Data Loading

In [ ]:
GTSRB_DATASET_NAME = 'meowmeowmeowmeowmeow/gtsrb-german-traffic-sign'
csv_paths = {
    'train.csv': None,
    'test.csv': None,
    'meta.csv': None,
}

download_dir = Path(kagglehub.dataset_download(GTSRB_DATASET_NAME))
for csv in download_dir.rglob('*.csv'):
    if csv.name.lower() in csv_paths:
        csv_paths[csv.name.lower()] = csv

train_csv = pd.read_csv(str(csv_paths['train.csv']))
train_df, val_df = train_test_split(
    train_csv,
    test_size=0.2,
    shuffle=True,
    stratify=train_csv.ClassId,
    random_state=SEED,
)
test_df = pd.read_csv(str(csv_paths['test.csv']))

prohibitory_sign_classes = [i for i in range(1, 11)]
prohibitory_sign_classes.remove(6)

prohib_train_df = train_df[train_df.ClassId.isin(prohibitory_sign_classes)].reset_index(drop=True)
prohib_test_df = test_df[test_df.ClassId.isin(prohibitory_sign_classes)].reset_index(drop=True)

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array(prohibitory_sign_classes),
    y=prohib_train_df.ClassId.values,
)
class_weights_dict = dict(zip(prohibitory_sign_classes, weights))
weights_tensor = torch.tensor(list(class_weights_dict.values()), dtype=torch.float, device=device)
class_mapping = {raw_id: i for i, raw_id in enumerate(sorted(prohibitory_sign_classes))}

print('Kaggle dataset directory:', download_dir)
print(f'Test examples: {len(prohib_test_df)}')

In [ ]:
class TrafficSignDataset(Dataset):
    def __init__(self, data_dir, image_file_paths, targets, transformer, class_mapping):
        if len(targets) != len(image_file_paths):
            raise AssertionError('Must have equal number of labels to images')

        self.data_dir = data_dir
        self.image_file_paths = list(image_file_paths)
        self.targets = list(targets)
        self.transformer = transformer
        self.class_mapping = class_mapping

    def __len__(self):
        return len(self.image_file_paths)

    def __getitem__(self, idx):
        img_path = os.path.join(self.data_dir, self.image_file_paths[idx])
        image = Image.open(img_path).convert('RGB')
        transformed_img = self.transformer(image)
        label = self.class_mapping[self.targets[idx]]
        target = {'label': torch.tensor(label, dtype=torch.long)}
        return transformed_img, target, str(img_path)


def collate_data(return_values):
    images = [item[0] for item in return_values]
    targets = [item[1] for item in return_values]
    paths = [item[2] for item in return_values]
    return images, targets, paths


transform_pipeline = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_dataset = TrafficSignDataset(
    download_dir,
    prohib_test_df.Path,
    prohib_test_df.ClassId,
    transform_pipeline,
    class_mapping,
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=collate_data,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSIST_WORKERS,
    generator=tor_gen,
)

loss_fn = CrossEntropyLoss(weight=weights_tensor)
print(f'Test dataloader batches: {len(test_dataloader)}')

## Model Reconstruction

In [ ]:
class MobileNetGeneralist(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base_model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        self.backbone = base_model.features
        self.pool = base_model.avgpool
        self.out_channels = base_model.classifier[0].in_features
        self.classifier = nn.Sequential(
            nn.Linear(self.out_channels, num_classes),
            nn.LayerNorm(num_classes),
        )

    def forward(self, x, return_features=False):
        x = self.backbone(x)
        features = self.pool(x).flatten(1)
        logits = self.classifier(features)
        if return_features:
            return logits, features
        return logits


class ConvNeXtTinyGeneralist(nn.Module):
    def __init__(self, num_classes, weights=models.ConvNeXt_Tiny_Weights.DEFAULT, dropout=0.1):
        super().__init__()
        base_model = models.convnext_tiny(weights=weights)
        self.backbone = base_model.features
        self.pool = base_model.avgpool
        self.out_channels = base_model.classifier[2].in_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.out_channels),
            nn.Dropout(dropout),
            nn.Linear(self.out_channels, num_classes),
        )

    def forward(self, x, return_features=False):
        x = self.backbone(x)
        features = self.pool(x).flatten(1)
        logits = self.classifier(features)
        if return_features:
            return logits, features
        return logits


class MlpRouter(nn.Module):
    def __init__(self, input_size, num_experts):
        super().__init__()
        self.num_experts = num_experts
        self.gating = nn.Linear(input_size, num_experts, bias=False)

    def forward(self, x):
        logits = self.gating(x)
        if self.training:
            logits = logits + torch.randn_like(logits) * (1.0 / self.num_experts)
        return logits


class LayeredRouter(nn.Module):
    def __init__(self, input_size, num_experts):
        super().__init__()
        self.num_experts = num_experts
        self.ff1 = nn.Linear(input_size, input_size * 2)
        self.ff2 = nn.Linear(input_size * 2, num_experts, bias=False)

    def forward(self, x):
        logits = self.ff2(self.ff1(x))
        if self.training:
            logits = logits + torch.randn_like(logits) * (1.0 / self.num_experts)
        return logits


class MlpExpert(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout=0.1):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, output_size),
        )

    def forward(self, x):
        return self.network(x)


class TransformerExpert(nn.Module):
    def __init__(self, in_features, d_model, nhead, output_size):
        super().__init__()
        if in_features % d_model != 0:
            raise ValueError('d_model must perfectly divide in_features')
        self.num_patches = in_features // d_model
        self.d_model = d_model
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, d_model))
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True),
            num_layers=1,
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, output_size),
        )

    def forward(self, x):
        batch_size = x.shape[0]
        x = x.view(batch_size, self.num_patches, self.d_model)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) + self.pos_embed
        x = self.transformer(x)
        cls_output = x[:, 0, :]
        return self.classifier(cls_output)


class ResidualGatedMlpExpert(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_size),
            nn.Linear(input_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.block1 = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size * 2),
            nn.GLU(dim=1),
            nn.Dropout(dropout),
        )
        self.block2 = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size * 2),
            nn.GLU(dim=1),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.input_proj(x)
        x = x + self.block1(x)
        x = x + self.block2(x)
        return self.classifier(x)


class JointEmbeddingRouterExperts(GeneralistRouterExperts):
    def __init__(self, *args, feature_key='projections', **kwargs):
        super().__init__(*args, **kwargs)
        self.feature_key = feature_key
        self._last_joint_embedding_outputs = None

    def reset_routing_state(self):
        super().reset_routing_state()
        self._last_joint_embedding_outputs = None

    def get_joint_embedding_loss(self, labels):
        if self._last_joint_embedding_outputs is None:
            param = next(self.parameters())
            return torch.zeros((), device=param.device, dtype=param.dtype)

        if self.feature_key == 'embeddings':
            return self.generalist.compute_contrastive_loss(
                labels=labels,
                embeddings=self._last_joint_embedding_outputs['embeddings'],
            )

        return self.generalist.compute_contrastive_loss(
            labels=labels,
            projections=self._last_joint_embedding_outputs['projections'],
        )

    def forward(self, x):
        encoded = self.generalist.backbone.encode(x)
        generalist_logits = encoded['logits']
        features = encoded[self.feature_key]
        if features.ndim != 2:
            features = torch.flatten(features, 1)

        router_logits = self.router(features)
        router_probs = F.softmax(router_logits, dim=1)
        topk_probs, topk_indices = torch.topk(router_probs, k=self.k, dim=1)
        topk_weights = topk_probs / topk_probs.sum(dim=1, keepdim=True).clamp_min(1e-8)

        experts_logits = torch.zeros_like(generalist_logits, device=features.device, dtype=features.dtype)
        for expert_idx, expert in enumerate(self.experts):
            selected_rows, selected_slots = torch.where(topk_indices == expert_idx)
            if selected_rows.numel() == 0:
                continue
            expert_features = features[selected_rows]
            expert_logits = expert(expert_features)
            expert_weights = topk_weights[selected_rows, selected_slots].unsqueeze(1)
            experts_logits[selected_rows] += expert_logits * expert_weights

        self._last_router_probs = router_probs.detach()
        self._last_topk_indices = topk_indices.detach()
        self._last_aux_loss = self._compute_auxiliary_loss(router_probs, topk_indices)
        self._last_joint_embedding_outputs = {
            key: value
            for key, value in encoded.items()
            if isinstance(value, torch.Tensor)
        }
        return (generalist_logits + experts_logits) / self.temperature


def strip_trailing_date_suffix(value):
    return re.sub(r'_\d{4}-\d{2}-\d{2}$', '', str(value))


def extract_model_name(checkpoint_path: Path):
    return strip_trailing_date_suffix(checkpoint_path.stem)


def build_expert_factory(expert_name, generalist_out_dim):
    def expert_factory():
        if expert_name == 'MLP Experts':
            return MlpExpert(
                input_size=generalist_out_dim,
                hidden_size=HIDDEN_EXPERT_SIZE,
                output_size=TOTAL_CLASSES,
                dropout=DROPOUT,
            )
        if expert_name == 'Transformer Experts':
            return TransformerExpert(
                in_features=generalist_out_dim,
                d_model=TFE_D_MODEL,
                nhead=TFE_NHEAD,
                output_size=TOTAL_CLASSES,
            )
        if expert_name == 'ResMLP Experts':
            return ResidualGatedMlpExpert(
                input_size=generalist_out_dim,
                hidden_size=HIDDEN_EXPERT_SIZE,
                output_size=TOTAL_CLASSES,
                dropout=DROPOUT,
            )
        raise KeyError(f'Unknown expert: {expert_name}')

    return expert_factory


def build_model_from_name(model_name: str):
    parts = str(model_name).split(' + ')
    if len(parts) != 3:
        raise ValueError(f'Expected model name in the form Generalist + Router + Expert, got: {model_name}')

    generalist_name, router_name, expert_name = parts

    if generalist_name == 'MobileNet':
        generalist = MobileNetGeneralist(num_classes=TOTAL_CLASSES)
        generalist_out_dim = generalist.out_channels
        router_cls = MlpRouter if router_name == 'MLP Router' else LayeredRouter
        router = router_cls(input_size=generalist_out_dim, num_experts=NUM_EXPERTS)
        return GeneralistRouterExperts(
            generalist=generalist,
            router=router,
            expert_factory=build_expert_factory(expert_name, generalist_out_dim),
            num_experts=NUM_EXPERTS,
            num_classes=TOTAL_CLASSES,
            k=TOP_K,
            lambda_aux=LAMBDA_AUX,
        )

    if generalist_name == 'ConvNeXt':
        generalist = ConvNeXtTinyGeneralist(num_classes=TOTAL_CLASSES)
        generalist_out_dim = generalist.out_channels
        router_cls = MlpRouter if router_name == 'MLP Router' else LayeredRouter
        router = router_cls(input_size=generalist_out_dim, num_experts=NUM_EXPERTS)
        return GeneralistRouterExperts(
            generalist=generalist,
            router=router,
            expert_factory=build_expert_factory(expert_name, generalist_out_dim),
            num_experts=NUM_EXPERTS,
            num_classes=TOTAL_CLASSES,
            k=TOP_K,
            lambda_aux=LAMBDA_AUX,
        )

    if generalist_name == 'JE MobileNet Small':
        generalist = JointEmbeddingModule(
            num_classes=TOTAL_CLASSES,
            backbone='mobilenet_v3_small',
            embedding_dim=JE_EMBEDDING_DIM,
            projection_dim=JE_PROJECTION_DIM,
            pretrained=True,
            temperature=JE_TEMPERATURE,
        )
        generalist_out_dim = JE_PROJECTION_DIM if JE_FEATURE_KEY == 'projections' else JE_EMBEDDING_DIM
        router_cls = MlpRouter if router_name == 'MLP Router' else LayeredRouter
        router = router_cls(input_size=generalist_out_dim, num_experts=NUM_EXPERTS)
        return JointEmbeddingRouterExperts(
            generalist=generalist,
            router=router,
            expert_factory=build_expert_factory(expert_name, generalist_out_dim),
            num_experts=NUM_EXPERTS,
            num_classes=TOTAL_CLASSES,
            k=TOP_K,
            lambda_aux=LAMBDA_AUX,
            feature_key=JE_FEATURE_KEY,
        )

    if generalist_name == 'JE ConvNeXt Tiny':
        generalist = JointEmbeddingModule(
            num_classes=TOTAL_CLASSES,
            backbone='convnext_tiny',
            embedding_dim=JE_EMBEDDING_DIM,
            projection_dim=JE_PROJECTION_DIM,
            pretrained=True,
            temperature=JE_TEMPERATURE,
        )
        generalist_out_dim = JE_PROJECTION_DIM if JE_FEATURE_KEY == 'projections' else JE_EMBEDDING_DIM
        router_cls = MlpRouter if router_name == 'MLP Router' else LayeredRouter
        router = router_cls(input_size=generalist_out_dim, num_experts=NUM_EXPERTS)
        return JointEmbeddingRouterExperts(
            generalist=generalist,
            router=router,
            expert_factory=build_expert_factory(expert_name, generalist_out_dim),
            num_experts=NUM_EXPERTS,
            num_classes=TOTAL_CLASSES,
            k=TOP_K,
            lambda_aux=LAMBDA_AUX,
            feature_key=JE_FEATURE_KEY,
        )

    raise KeyError(f'Unsupported generalist name: {generalist_name}')

In [ ]:
def to_json_compatible(value):
    if isinstance(value, dict):
        return {str(key): to_json_compatible(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_json_compatible(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    return value


def add_model_component_columns(df: pd.DataFrame):
    if 'Model' not in df.columns:
        raise KeyError("Expected a 'Model' column in the comparison dataframe.")

    parts = df['Model'].astype(str).str.split(r'\s+\+\s+', expand=True)
    if parts.shape[1] < 3:
        raise ValueError("Expected model names in the form 'Generalist + Router + Expert'.")

    out = df.copy()
    out['Generalist'] = parts[0]
    out['Router'] = parts[1]
    out['Expert'] = parts[2]
    return out


def normalize_backbone_label(generalist_name: str):
    name = str(generalist_name).strip()
    lowered = name.lower()
    if lowered.startswith('je mobilenet'):
        return 'JE MobileNet'
    if lowered.startswith('je convnext'):
        return 'JE ConvNeXt'
    if lowered.startswith('mobilenet'):
        return 'MobileNet'
    if lowered.startswith('convnext'):
        return 'ConvNeXt'
    return np.nan


def flatten_multiindex_columns(df: pd.DataFrame):
    flattened = []
    for col in df.columns:
        if isinstance(col, tuple):
            flattened.append(' '.join(str(part) for part in col if part).strip())
        else:
            flattened.append(str(col))
    result = df.copy()
    result.columns = flattened
    return result


def summarize_component_performance(df: pd.DataFrame, component_col: str, metrics: list[str]):
    available_metrics = [metric for metric in metrics if metric in df.columns]
    if not available_metrics:
        return pd.DataFrame()

    summary = df.groupby(component_col)[available_metrics].agg(['mean']).reset_index()
    summary = flatten_multiindex_columns(summary)
    sort_column = f'{available_metrics[0]} mean'
    if sort_column in summary.columns:
        summary = summary.sort_values(sort_column, ascending=False).reset_index(drop=True)
    return summary


def build_component_delta_table(df: pd.DataFrame, component_col: str, baseline_name: str, compare_names: list[str], fixed_cols: list[str], metrics: list[str]):
    available_metrics = [metric for metric in metrics if metric in df.columns]
    if not available_metrics:
        return pd.DataFrame()

    rows = []
    for compare_name in compare_names:
        subset = df[df[component_col].isin([baseline_name, compare_name])].copy()
        if subset[component_col].nunique() < 2:
            continue

        pivot = subset.pivot_table(index=fixed_cols, columns=component_col, values=available_metrics, aggfunc='first')
        if baseline_name not in pivot.columns.get_level_values(1):
            continue
        if compare_name not in pivot.columns.get_level_values(1):
            continue

        delta_df = pd.DataFrame(index=pivot.index).reset_index()
        delta_df[component_col] = compare_name
        for metric in available_metrics:
            baseline_key = (metric, baseline_name)
            compare_key = (metric, compare_name)
            if baseline_key not in pivot.columns or compare_key not in pivot.columns:
                continue
            baseline_values = pivot[baseline_key].to_numpy()
            compare_values = pivot[compare_key].to_numpy()
            delta_df[f'{metric} [{baseline_name}]'] = baseline_values
            delta_df[f'{metric} [{compare_name}]'] = compare_values
            delta_df[f'{metric} Delta ({compare_name} - {baseline_name})'] = compare_values - baseline_values

        rows.append(delta_df)

    if not rows:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)


def run_paired_component_tests(df: pd.DataFrame, component_col: str, baseline_name: str, compare_names: list[str], fixed_cols: list[str], metrics: list[str]):
    available_metrics = [metric for metric in metrics if metric in df.columns]
    if not available_metrics:
        return pd.DataFrame()

    rows = []
    raw_p_values = []
    p_value_row_indices = []

    for compare_name in compare_names:
        subset = df[df[component_col].isin([baseline_name, compare_name])].copy()
        if subset[component_col].nunique() < 2:
            continue

        pivot = subset.pivot_table(index=fixed_cols, columns=component_col, values=available_metrics, aggfunc='first')
        if baseline_name not in pivot.columns.get_level_values(1):
            continue
        if compare_name not in pivot.columns.get_level_values(1):
            continue

        for metric in available_metrics:
            baseline_key = (metric, baseline_name)
            compare_key = (metric, compare_name)
            if baseline_key not in pivot.columns or compare_key not in pivot.columns:
                continue

            paired = pivot[[baseline_key, compare_key]].dropna()
            if paired.empty:
                continue

            baseline_values = paired[baseline_key].to_numpy(dtype=float)
            compare_values = paired[compare_key].to_numpy(dtype=float)
            deltas = compare_values - baseline_values
            nonzero_deltas = deltas[~np.isclose(deltas, 0.0)]

            p_value = np.nan
            test_name = 'Wilcoxon signed-rank'
            if len(nonzero_deltas) == 0:
                p_value = 1.0
                test_name = 'All deltas are zero'
            elif len(nonzero_deltas) >= 2:
                try:
                    _, p_value = wilcoxon(nonzero_deltas, alternative='two-sided', zero_method='wilcox')
                except ValueError:
                    p_value = np.nan
                    test_name = 'Wilcoxon unavailable'
            else:
                test_name = 'Insufficient non-zero pairs'

            row = {
                component_col: compare_name,
                'Metric': metric,
                'Paired Configurations': len(deltas),
                f'Mean {baseline_name}': baseline_values.mean(),
                f'Mean {compare_name}': compare_values.mean(),
                'Mean Delta': deltas.mean(),
                'Median Delta': np.median(deltas),
                'Std Delta': deltas.std(ddof=0),
                'Raw P Value': p_value,
                'Test': test_name,
            }
            rows.append(row)
            if not np.isnan(p_value):
                raw_p_values.append(p_value)
                p_value_row_indices.append(len(rows) - 1)

    if not rows:
        return pd.DataFrame()

    result = pd.DataFrame(rows)
    result['FDR BH P Value'] = np.nan
    result['Significant @ 0.05'] = False
    if raw_p_values:
        _, corrected, _, _ = multipletests(raw_p_values, method='fdr_bh')
        for row_index, corrected_p in zip(p_value_row_indices, corrected):
            result.loc[row_index, 'FDR BH P Value'] = corrected_p
            result.loc[row_index, 'Significant @ 0.05'] = bool(corrected_p < 0.05)

    return result.sort_values([component_col, 'Metric']).reset_index(drop=True)


def style_component_table(df: pd.DataFrame):
    if df.empty:
        return df

    formatters = {}
    for column in df.columns:
        if pd.api.types.is_numeric_dtype(df[column]):
            if 'count' in column.lower() or 'paired configurations' in column.lower():
                formatters[column] = '{:.0f}'
            elif 'Cost Proxy' in column or 'Num Parameters' in column:
                formatters[column] = '{:.0f}'
            else:
                formatters[column] = '{:.4f}'

    return df.style.format(formatters)


def collect_test_gre_metrics(cil_model, eval_dataloader, num_classes=TOTAL_CLASSES, device=device):
    model = cil_model.model
    if not isinstance(model, GeneralistRouterExperts):
        return {}

    entropy_metrics = router_entropy(model, eval_dataloader, device=device)
    expert_summary = expert_metrics(model, eval_dataloader, device=device)
    ece = expected_calibration_error(
        model,
        num_classes=num_classes,
        val_data=eval_dataloader,
        device=device,
    )
    params = model.get_model_parameters_by_component()
    expected_calls = expert_summary['Expected Expert Calls']

    return {
        'Test Router Entropy': entropy_metrics['Mean Router Entropy'],
        'Test Normalized Router Entropy': entropy_metrics['Normalized Router Entropy'],
        'Test Router Entropy Std': entropy_metrics['Std Router Entropy'],
        'Test Avg Router Prob': entropy_metrics['Avg Router Prob'],
        'Test ECE': ece,
        'Test Expected Expert Calls': expected_calls,
        'Test Expert Utilization Ratio': expert_summary['Expert Selection Ratio'],
        'Test Cost Proxy': cost_proxy(expected_calls, params),
    }


def evaluate_checkpoint(checkpoint_path: Path):
    model_name = extract_model_name(checkpoint_path)
    model_instance = build_model_from_name(model_name)
    cil_model = CILComputerVisionModel.load(model_instance, filename=str(checkpoint_path), device=device)
    cil_model.model.to(device)
    test_loss, test_metrics = cil_model.evaluate(test_dataloader, loss_fn, apply_class_masking=USE_CLASS_MASKING)

    result = {
        'Model': model_name,
        'Source Checkpoint': str(checkpoint_path),
        'Test Loss': test_loss,
        'Test Macro F1': test_metrics['Macro F1'],
        'Test Micro F1': test_metrics['Micro F1'],
        'Test Weighted F1': test_metrics['Weighted F1'],
        'Num Parameters': sum(parameter.numel() for parameter in cil_model.model.parameters()),
    }
    result.update(collect_test_gre_metrics(cil_model, test_dataloader, device=device))

    output_path = TEST_RESULTS_DIR / f'test_{model_name}.json'
    with output_path.open('w', encoding='utf-8') as handle:
        json.dump(to_json_compatible(result), handle, indent=4)

    return result, cil_model, output_path

## Load and Evaluate Checkpoints

In [ ]:
checkpoint_paths = sorted(MODELS_DIR.glob('*.pth'))
if not checkpoint_paths:
    raise FileNotFoundError(f'No .pth checkpoints were found in {MODELS_DIR.resolve()}')

pd.DataFrame({
    'Checkpoint File': [path.name for path in checkpoint_paths],
    'Model Name': [extract_model_name(path) for path in checkpoint_paths],
})

In [ ]:
test_results = []
loaded_wrappers = {}
loaded_models = {}
saved_test_json_paths = []

for checkpoint_path in checkpoint_paths:
    print(f'Evaluating {checkpoint_path.name}...')
    result, cil_model, output_path = evaluate_checkpoint(checkpoint_path)
    test_results.append(result)
    loaded_wrappers[result['Model']] = cil_model
    loaded_models[result['Model']] = cil_model.model
    saved_test_json_paths.append(output_path)

test_results_df = pd.DataFrame(test_results)
test_results_df

In [ ]:
test_results_df = add_model_component_columns(test_results_df)
test_results_df['Backbone'] = test_results_df['Generalist'].map(normalize_backbone_label)
test_results_df['Joint Embedding'] = np.where(test_results_df['Backbone'].astype(str).str.startswith('JE '), 'JE', 'No JE')
test_results_df['Backbone'] = pd.Categorical(test_results_df['Backbone'], categories=BACKBONE_ORDER, ordered=True)
test_results_df['Configuration'] = test_results_df['Router'] + ' + ' + test_results_df['Expert']

summary_columns = [
    column for column in [
        'Model',
        'Backbone',
        'Router',
        'Expert',
        'Test Loss',
        'Test Macro F1',
        'Test Micro F1',
        'Test Weighted F1',
        'Test ECE',
        'Test Router Entropy',
        'Test Expected Expert Calls',
        'Test Cost Proxy',
        'Num Parameters',
    ] if column in test_results_df.columns
]
summary_df = test_results_df.loc[:, summary_columns].copy()
summary_df = summary_df.sort_values(by=['Test Micro F1', 'Test Macro F1'], ascending=False).reset_index(drop=True)
display(summary_df.style.format({
    'Test Loss': '{:.4f}',
    'Test Macro F1': '{:.4f}',
    'Test Micro F1': '{:.4f}',
    'Test Weighted F1': '{:.4f}',
    'Test ECE': '{:.4f}',
    'Test Router Entropy': '{:.4f}',
    'Test Expected Expert Calls': '{:.4f}',
    'Test Cost Proxy': '{:.0f}',
    'Num Parameters': '{:,.0f}',
}))

pd.DataFrame({'Saved Test JSON': [str(path) for path in saved_test_json_paths]})

In [ ]:
component_metrics = [
    metric
    for metric in [
        'Test Macro F1',
        'Test Micro F1',
        'Test Weighted F1',
        'Test ECE',
        'Test Router Entropy',
        'Test Expected Expert Calls',
        'Test Cost Proxy',
        'Num Parameters',
    ]
    if metric in test_results_df.columns
]

print('Model count by backbone, router, and expert combination')
display(pd.crosstab(test_results_df['Backbone'], [test_results_df['Router'], test_results_df['Expert']]))

backbone_summary_df = summarize_component_performance(test_results_df, 'Backbone', component_metrics)
display(style_component_table(backbone_summary_df))

je_summary_df = summarize_component_performance(test_results_df, 'Joint Embedding', component_metrics)
display(style_component_table(je_summary_df))

backbone_compare_names = [name for name in BACKBONE_ORDER if name in set(test_results_df['Backbone'].astype(str)) and name != BASELINE_BACKBONE]
paired_metrics = [metric for metric in ['Test Macro F1', 'Test Micro F1', 'Test Weighted F1', 'Test ECE', 'Test Router Entropy', 'Num Parameters'] if metric in test_results_df.columns]
backbone_delta_df = build_component_delta_table(test_results_df, 'Backbone', BASELINE_BACKBONE, backbone_compare_names, ['Router', 'Expert'], paired_metrics)
display(style_component_table(backbone_delta_df))

backbone_significance_df = run_paired_component_tests(test_results_df, 'Backbone', BASELINE_BACKBONE, backbone_compare_names, ['Router', 'Expert'], paired_metrics)
display(style_component_table(backbone_significance_df))

In [ ]:
if len(loaded_models) >= 2:
    bootstrap_results, bootstrap_model_performance = bootstrap_performance_diff(
        test_dataloader,
        loaded_models,
        device=device,
        resamples=BS_RESAMPLES,
    )
    bootstrap_results_df = pd.DataFrame(bootstrap_results)
    bootstrap_model_performance_df = pd.DataFrame(bootstrap_model_performance)

    if not bootstrap_results_df.empty:
        display(bootstrap_results_df.style.format({
            'Mean Delta (F1_A - F1_B)': '{:.4f}',
            '95% CI Lower': '{:.4f}',
            '95% CI Upper': '{:.4f}',
            'p_value': '{:.4f}',
            'BH_p_value': '{:.4f}',
        }))

    if not bootstrap_model_performance_df.empty:
        display(bootstrap_model_performance_df.style.format({
            'Mean F1': '{:.4f}',
            'Std F1': '{:.4f}',
        }))
else:
    print('At least two models are required to run bootstrap pairwise comparisons.')

## Separate Figures

These figures mirror the backbone comparison notebook, but use held-out test-set metrics instead of serialized continual-learning validation metrics.

In [ ]:
sns.set_theme(style='whitegrid')
palette = {
    'MobileNet': '#4C78A8',
    'ConvNeXt': '#F58518',
    'JE MobileNet': '#54A24B',
    'JE ConvNeXt': '#E45756',
}

backbone_plot_metrics = [
    metric for metric in [
        'Test Micro F1',
        'Test Macro F1',
        'Test Weighted F1',
        'Test ECE',
        'Test Router Entropy',
        'Num Parameters',
    ] if metric in test_results_df.columns
]

if {'Num Parameters', 'Test Micro F1', 'Backbone', 'Router'}.issubset(test_results_df.columns):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=test_results_df,
        x='Num Parameters',
        y='Test Micro F1',
        hue='Backbone',
        style='Router',
        palette=palette,
        s=120,
    )
    plt.title('Test Micro F1 vs Parameter Count by Backbone')
    plt.xlabel('Num Parameters')
    plt.ylabel('Test Micro F1')
    plt.tight_layout()
    plt.show()

In [ ]:
for metric in [name for name in ['Test Micro F1', 'Test Macro F1', 'Test ECE'] if name in test_results_df.columns]:
    pivot = test_results_df.pivot_table(index='Configuration', columns='Backbone', values=metric, aggfunc='mean')
    pivot = pivot.reindex(columns=BACKBONE_ORDER)
    plt.figure(figsize=(9, max(4, len(pivot) * 0.6)))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu')
    plt.title(f'Mean {metric} by Router + Expert Configuration and Backbone')
    plt.xlabel('Backbone')
    plt.ylabel('Router + Expert Configuration')
    plt.tight_layout()
    plt.show()

In [ ]:
for metric in backbone_plot_metrics:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=test_results_df, x='Backbone', y=metric, order=BACKBONE_ORDER, palette=palette, fliersize=0)
    sns.stripplot(data=test_results_df, x='Backbone', y=metric, order=BACKBONE_ORDER, color='black', alpha=0.75, size=6)
    plt.title(f'Test Set Backbone Comparison: {metric}')
    plt.xlabel('Backbone')
    plt.ylabel(metric)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plot_df = test_results_df.sort_values('Test Micro F1', ascending=False)
sns.barplot(data=plot_df, x='Test Micro F1', y='Model', hue='Joint Embedding')
plt.title('Test Micro F1 by Model')
plt.xlabel('Test Micro F1')
plt.ylabel('Model')
plt.tight_layout()
plt.show()